# Claude Code + MCP  →  Cortex Agents + CoWork
## Notebook 01 — Shared Foundation (Part 0)

**Scenario (client-agnostic):** a B2B sales-intelligence / go-to-market (GTM) SaaS company scores every
sales-rep email with AI and mines winning email patterns. Over four notebooks we migrate that workload
from an **external brain** (Claude Code over the Snowflake-managed MCP server) to an **in-data-plane
multi-agent architecture** (Cortex Agents + CoWork) — and prove lower latency, lower cost, stronger
governance, and built-in observability.

This notebook establishes the shared foundation **both** paths reuse: the data, the semantic view
(Cortex Analyst), the Cortex Search service, and a governed SQL tool.

### What you'll do
1. Connect and tour the synthetic GTM email corpus.
2. Confirm the three reusable tools: Cortex Analyst (semantic view), Cortex Search, governed UDF.
3. Run **Checkpoint 0** — all must PASS before Part A.

**Prerequisite:** run `lab/setup.sql` once before this notebook.

---
## Section 1: Connect

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.sql("USE DATABASE GTMAGENTS").collect()
session.sql("USE SCHEMA GTMAGENTS.DEMO").collect()
session.sql("USE WAREHOUSE GTMAGENTS_WH").collect()
print("Connected to GTMAGENTS.DEMO")

---
## Section 2: Tour the data

Four tables: `REPS`, `EMAILS` (subject/body text generated across good / mixed / poor quality tiers),
`OUTCOMES` (deal progression + revenue), and `EMAIL_FRAMEWORK` (the best-practice rubric).
The latent quality tier drives engagement, so intent scoring (Part B) will be genuinely learnable from the text.

In [ ]:
# Row counts
session.sql("""
    SELECT 'REPS' AS tbl, COUNT(*) AS rows FROM REPS
    UNION ALL SELECT 'EMAILS', COUNT(*) FROM EMAILS
    UNION ALL SELECT 'OUTCOMES', COUNT(*) FROM OUTCOMES
    UNION ALL SELECT 'EMAIL_FRAMEWORK', COUNT(*) FROM EMAIL_FRAMEWORK
""").show()

In [ ]:
# Quality spread — reply/win rates separate cleanly by tier (1=poor, 2=mixed, 3=good)
session.sql("""
    SELECT e.quality_tier,
           COUNT(*) AS emails,
           ROUND(AVG(IFF(e.replied,1,0)),3) AS reply_rate,
           ROUND(AVG(IFF(o.won,1,0)),3)     AS win_rate,
           SUM(o.revenue)                    AS revenue
    FROM EMAILS e JOIN OUTCOMES o ON o.email_id = e.id
    GROUP BY e.quality_tier ORDER BY e.quality_tier
""").show()

In [ ]:
# Sample one email per tier so you can see the text quality difference
session.sql("""
    SELECT quality_tier, subject, body FROM EMAILS
    QUALIFY ROW_NUMBER() OVER (PARTITION BY quality_tier ORDER BY RANDOM()) = 1
    ORDER BY quality_tier
""").show(max_width=200)

---
## Section 3: Tool 1 — Cortex Analyst (semantic view)

`EMAIL_GTM_SV` is the governed semantic view over emails + reps + outcomes. Cortex Analyst turns a natural-
language question into a `SEMANTIC_VIEW(...)` query. Below is the SQL Analyst generates for
*"What is the reply rate and win rate by region?"* — the **same** semantic view is reused by the MCP server
(Part A) and the Recommendation agent (Part B), so business definitions never drift between the two worlds.

In [ ]:
session.sql("""
    SELECT * FROM SEMANTIC_VIEW(
        EMAIL_GTM_SV
        DIMENSIONS reps.region
        METRICS reply_rate, win_rate, total_revenue
    ) ORDER BY total_revenue DESC
""").show()

---
## Section 4: Tool 2 & 3 — Cortex Search + governed SQL tool

`FRAMEWORK_SEARCH` retrieves best-practice rubric text (used by the Coaching agent to rewrite weak emails).
`GTM_TEAM_PERFORMANCE(region)` is a **governed** read-only UDF that returns curated aggregate GTM metrics as a
single JSON cell — never raw rows — honoring the Cortex custom-tool contract and the least-privilege story.

In [ ]:
# Cortex Search preview — retrieve the rubric principles most relevant to a weak email
session.sql("""
    SELECT PARSE_JSON(
      SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'GTMAGENTS.DEMO.FRAMEWORK_SEARCH',
        '{"query": "email has no clear call to action and is generic", "limit": 3}'
      ))['results'] AS results
""").show(max_width=200)

In [ ]:
# Governed SQL tool — aggregate team performance for one region (returns a JSON array, single cell)
session.sql("SELECT GTM_TEAM_PERFORMANCE('AMER') AS amer_performance").show(max_width=200)

---
## ✅ Checkpoint 0

All checks must PASS before Part A (`gtm-02-before-mcp`).

In [ ]:
checks = []

rc = {r['TBL']: r['ROWS'] for r in session.sql("""
    SELECT 'REPS' AS tbl, COUNT(*) AS rows FROM REPS
    UNION ALL SELECT 'EMAILS', COUNT(*) FROM EMAILS
    UNION ALL SELECT 'OUTCOMES', COUNT(*) FROM OUTCOMES
    UNION ALL SELECT 'EMAIL_FRAMEWORK', COUNT(*) FROM EMAIL_FRAMEWORK
""").collect()}
checks.append(('REPS > 0', rc.get('REPS',0) > 0))
checks.append(('EMAILS in 2000-5000', 2000 <= rc.get('EMAILS',0) <= 5000))
checks.append(('OUTCOMES > 0', rc.get('OUTCOMES',0) > 0))
checks.append(('EMAIL_FRAMEWORK > 0', rc.get('EMAIL_FRAMEWORK',0) > 0))

# Cortex Analyst semantic view returns rows
analyst_rows = session.sql("""
    SELECT * FROM SEMANTIC_VIEW(EMAIL_GTM_SV DIMENSIONS reps.region METRICS reply_rate)
""").count()
checks.append(('Analyst semantic view returns rows', analyst_rows > 0))

# Governed tool executes and returns a non-empty array
tool_size = session.sql("SELECT ARRAY_SIZE(GTM_TEAM_PERFORMANCE('')) AS n").collect()[0]['N']
checks.append(('Governed SQL tool returns rows', (tool_size or 0) > 0))

# Cortex Search service is ready
svc = session.sql("SHOW CORTEX SEARCH SERVICES LIKE 'FRAMEWORK_SEARCH' IN SCHEMA GTMAGENTS.DEMO").count()
checks.append(('Cortex Search service exists', svc > 0))

print('CHECKPOINT 0')
for name, ok in checks:
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}")
overall = all(ok for _, ok in checks)
print()
print('RESULT:', 'PASS — proceed to Part A' if overall else 'FAIL — fix above before Part A')
assert overall, 'Checkpoint 0 failed'

---
## Summary

The shared foundation is live: synthetic GTM email data plus three governed, reusable tools
(Cortex Analyst semantic view, Cortex Search, governed UDF). Next, **`gtm-02-before-mcp`** exposes these
same tools through a Snowflake-managed MCP server so Claude Code can drive them as an external brain.